# Brain Tumor Detection using Deep Learning (CNNs)

This notebook implements a Convolutional Neural Network (CNN) to classify brain MRI scans into four categories:
- **Glioma**
- **Meningioma**
- **No Tumor**
- **Pituitary**

**Dataset:** [Brain Tumor MRI Dataset](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset) on Kaggle

## 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from sklearn.model_selection import train_test_split
import tensorflow as tf
import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

## 2. Dataset Setup

Define the path to the dataset and the tumor categories.

In [ ]:
data = '/kaggle/input/brain-tumor-mri-dataset/Training'
categories = ['glioma', 'meningioma', 'notumor', 'pituitary']

### 2.1 List Category Folders

In [ ]:
folds = [os.path.join(data, catg) for catg in categories]
folds

### 2.2 Count Images per Category

In [ ]:
for fold in folds:
    print(fold.split('/')[5], ':', len(os.listdir(fold)))

### 2.3 Visualize Class Distribution

In [ ]:
# Data extracted from output
labels = ['glioma', 'meningioma', 'notumor', 'pituitary']
values = [1321, 1339, 1595, 1457]

plt.figure(figsize=(8, 4))
plt.bar(labels, values, color='green')
plt.title('Number of Images per Tumor Class')
plt.xlabel('Tumor Type')
plt.ylabel('Number of Images')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### 2.4 Inspect Image Dimensions

Check the shape of a sample image from each category.

In [ ]:
def img_array_dim_all(folds, categories):
    for idx, folder in enumerate(folds):
        for img in os.listdir(folder):
            img_path = os.path.join(folder, img)
            img_array = cv2.imread(img_path)
            if img_array is not None:
                print(f"Category: {categories[idx]}")
                print(f"Dim: {img_array.shape}")
                print("-------------------")
                print(img_array)
                print("\n")
                break  # show only 1 sample per category

# Example usage
data = "/kaggle/input/brain-tumor-mri-dataset/Training"
categories = os.listdir(data)
folds = [os.path.join(data, catg) for catg in categories]

img_array_dim_all(folds, categories)

### 2.5 Visualize Sample Images per Category

In [ ]:
# Reset to defined categories (sorted)
categories = ['glioma', 'meningioma', 'notumor', 'pituitary']
folds = [os.path.join(data, catg) for catg in categories]

# Number of samples per category to display
samples_per_class = 5

plt.figure(figsize=(12, 8))

# Loop through categories
for idx, category in enumerate(categories):
    folder_path = os.path.join(data, category)
    images = os.listdir(folder_path)

    # Pick random images from this category
    sample_imgs = random.sample(images, min(samples_per_class, len(images)))

    for i, img_name in enumerate(sample_imgs):
        img_path = os.path.join(folder_path, img_name)
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Plot
        plt.subplot(len(categories), samples_per_class, idx * samples_per_class + i + 1)
        plt.imshow(img)
        plt.axis("off")
        plt.title(category)

plt.tight_layout()
plt.show()

## 3. Data Preprocessing

### 3.1 Load and Resize Images

All images are resized to 128×128 pixels and stored in arrays `X` and `y`.

In [ ]:
# Load dataset
img_size = 128
X = []
y = []

for idx, category in enumerate(categories):
    folder = os.path.join(data, category)
    for img_name in os.listdir(folder):
        img_path = os.path.join(folder, img_name)
        img = cv2.imread(img_path)
        if img is not None:
            img = cv2.resize(img, (img_size, img_size))
            X.append(img)
            y.append(idx)

### 3.2 Normalize Pixel Values

Scale pixel values from [0, 255] to [0, 1].

In [ ]:
X = np.array(X, dtype="float32") / 255.0
y = np.array(y)

### 3.3 One-Hot Encode Labels

In [ ]:
# One-hot encode labels
y = to_categorical(y, num_classes=len(categories))

### 3.4 Train/Test Split

Split the dataset into 80% training and 20% testing with stratification.

In [ ]:
# Train/test split
x_train, x_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", x_train.shape)
print("Testing samples:", x_test.shape)

## 4. Model Architecture

### 4.1 Build CNN

The CNN consists of:
- Two `Conv2D` layers with ReLU activation and stride-based downsampling
- `MaxPooling2D` for spatial reduction
- `Dropout` layers for regularization
- Fully connected `Dense` layers
- Final `softmax` output layer for 4-class classification

In [ ]:
# Build CNN
model = Sequential()

model.add(Conv2D(32, (5, 5), strides=(2, 2), padding="same", input_shape=(img_size, img_size, 3)))
model.add(Activation("relu"))

model.add(Conv2D(32, (5, 5), strides=(2, 2)))
model.add(Activation("relu"))

model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

model.add(Flatten())
model.add(Dense(512, activation="relu"))
model.add(Dropout(0.5))
model.add(Dense(len(categories), activation="softmax"))

### 4.2 Compile the Model

In [ ]:
opt = RMSprop(learning_rate=0.0005)

model.compile(
    loss="categorical_crossentropy",
    optimizer=opt,
    metrics=["accuracy"]
)

### 4.3 Model Summary

In [ ]:
# Check number of parameters
model.summary()

## 5. Training

Train the model for 15 epochs with a batch size of 32.

**Results summary:**
| Epoch | Train Accuracy | Val Accuracy |
|-------|---------------|--------------|
| 1     | 54.55%        | 73.49%       |
| 5     | 90.58%        | 90.55%       |
| 10    | 97.09%        | 93.26%       |
| 15    | 98.87%        | 94.05%       |

In [ ]:
# Train
history = model.fit(
    x_train,
    y_train,
    batch_size=32,
    epochs=15,
    validation_data=(x_test, y_test),
    shuffle=True
)

## 6. Evaluation

### 6.1 Test Accuracy

In [ ]:
# Evaluate
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"Test Accuracy: {test_acc:.4f}")

### 6.2 Classification Report & Confusion Matrix

**Results:**
| Class      | Precision | Recall | F1-Score | Support |
|------------|-----------|--------|----------|---------|
| pituitary  | 0.96      | 0.99   | 0.97     | 292     |
| notumor    | 0.97      | 0.97   | 0.97     | 319     |
| meningioma | 0.87      | 0.90   | 0.89     | 268     |
| glioma     | 0.96      | 0.89   | 0.92     | 264     |
| **Overall**| **0.94**  | **0.94**| **0.94**| 1143    |

In [ ]:
y_pred = model.predict(x_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

print(classification_report(y_true, y_pred_classes, target_names=categories))

cm = confusion_matrix(y_true, y_pred_classes)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=categories, yticklabels=categories)
plt.ylabel('True')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()

### 6.3 Visualize Sample Predictions

In [ ]:
plt.figure(figsize=(12, 8))
for i in range(9):
    idx = np.random.randint(0, len(x_test))
    plt.subplot(4, 4, i + 1)
    plt.imshow(x_test[idx])
    plt.title(f"True: {categories[y_true[idx]]}\nPred: {categories[y_pred_classes[idx]]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 7. Save the Model

In [ ]:
# Save the trained model
model.save('brain_tumor_cnn_model.h5')
print("Model saved as brain_tumor_cnn_model.h5")